# EMG Feature Clustering

Unsupervised K-Means clustering on the 192-dimensional hand-crafted EMG features.

**Data**: 5 subjects (03–07), ~280k windows, 192 features (8 per channel × 24 channels)  
**Goal**: Group windows into 8 clusters and evaluate how well they align with the 7 true gesture classes  
**Pipeline**:
1. Load all per-subject feature `.mat` files and combine
2. StandardScaler normalisation
3. PCA to 50 components (speed + de-noise)
4. K-Means (k=8)
5. Evaluation: silhouette score, cluster purity, confusion matrix, PCA-2D scatter

In [ ]:
import sys, os
import glob
import numpy as np
import scipy.io
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
    confusion_matrix,
)
from scipy.stats import mode

RANDOM_STATE = 42
N_CLUSTERS   = 8
GESTURE_NAMES = ["G1", "G2", "G3", "G6", "G7", "G8", "G9"]

## 1 — Load & combine all subjects

In [ ]:
FEATURE_DIR = os.path.join(
    os.path.dirname(os.path.dirname(os.path.abspath("__file__"))),
    "Data", "X", "feature_files"
)

# Fallback: resolve relative to notebook location
if not os.path.isdir(FEATURE_DIR):
    FEATURE_DIR = os.path.abspath(
        os.path.join("..", "..", "Data", "X", "feature_files")
    )

mat_files = sorted(glob.glob(os.path.join(FEATURE_DIR, "features_subject_*.mat")))
print(f"Found {len(mat_files)} feature file(s) in:\n  {FEATURE_DIR}\n")

X_all, y_all, subject_ids = [], [], []

for fpath in mat_files:
    data = scipy.io.loadmat(fpath)
    X_sub = data["X"].astype(np.float32)       # (n_windows, 192)
    y_sub = data["y"].squeeze().astype(int)    # (n_windows,)

    subject = os.path.basename(fpath).replace("features_subject_", "").replace(".mat", "")
    subject_ids.append(np.full(len(y_sub), int(subject), dtype=int))

    X_all.append(X_sub)
    y_all.append(y_sub)
    print(f"  Subject {subject}: X={X_sub.shape}, y={y_sub.shape}")

X = np.vstack(X_all)                           # (N_total, 192)
y = np.concatenate(y_all)                      # (N_total,)
subjects = np.concatenate(subject_ids)         # (N_total,)

print(f"\nCombined: X={X.shape}, y={y.shape}")
print(f"Class distribution: { {GESTURE_NAMES[i]: int((y==i).sum()) for i in range(7)} }")

## 2 — Normalise + PCA

In [ ]:
# StandardScaler: zero mean, unit variance per feature
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA: reduce to 50 components for speed while retaining most variance
N_PCA = 50
pca = PCA(n_components=N_PCA, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_.cumsum()[-1]
print(f"PCA {N_PCA} components explain {explained*100:.1f}% of variance")

# Also keep 2-D projection for visualisation
pca2 = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca2.fit_transform(X_scaled)
print(f"2-D PCA explains {pca2.explained_variance_ratio_.sum()*100:.1f}% of variance")

## 3 — K-Means clustering (k=8)

In [ ]:
# MiniBatchKMeans is much faster than full KMeans for ~280k samples
# Use n_init=10 restarts to reduce sensitivity to initialisation
kmeans = MiniBatchKMeans(
    n_clusters=N_CLUSTERS,
    n_init=10,
    random_state=RANDOM_STATE,
    batch_size=4096,
    max_iter=300,
    verbose=0,
)

print(f"Running MiniBatchKMeans with k={N_CLUSTERS} on X_pca {X_pca.shape} ...")
cluster_labels = kmeans.fit_predict(X_pca)
print("Done.")

unique, counts = np.unique(cluster_labels, return_counts=True)
print("\nCluster sizes:")
for c, n in zip(unique, counts):
    print(f"  Cluster {c}: {n:,} windows ({n/len(cluster_labels)*100:.1f}%)")

## 4 — Evaluation

In [ ]:
# Subsample for silhouette (expensive on large datasets)
SILHOUETTE_SAMPLE = 10_000
rng = np.random.default_rng(RANDOM_STATE)
idx = rng.choice(len(cluster_labels), size=SILHOUETTE_SAMPLE, replace=False)
sil = silhouette_score(X_pca[idx], cluster_labels[idx], metric="euclidean")

ari = adjusted_rand_score(y, cluster_labels)
nmi = normalized_mutual_info_score(y, cluster_labels)

print(f"Silhouette Score (sample={SILHOUETTE_SAMPLE}): {sil:.4f}  (higher is better, max=1)")
print(f"Adjusted Rand Index               : {ari:.4f}  (1=perfect match with true labels)")
print(f"Normalised Mutual Information      : {nmi:.4f}  (1=perfect)")

# Cluster purity: for each cluster, dominant true label
purity_total = 0
print("\nCluster purity (dominant gesture per cluster):")
print(f"  {'Cluster':>8} {'Dominant gesture':>18} {'Purity':>8} {'Size':>8}")
print("  " + "-"*48)
for c in range(N_CLUSTERS):
    mask = cluster_labels == c
    labels_in_cluster = y[mask]
    dom_label = int(mode(labels_in_cluster, keepdims=True).mode[0])
    purity = (labels_in_cluster == dom_label).sum() / mask.sum()
    purity_total += (labels_in_cluster == dom_label).sum()
    print(f"  {c:>8} {GESTURE_NAMES[dom_label]:>18} {purity:>7.1%} {mask.sum():>8,}")

overall_purity = purity_total / len(y)
print(f"\nOverall cluster purity: {overall_purity:.4f} ({overall_purity*100:.1f}%)")

## 5 — Visualisations

In [ ]:
# ── PCA-2D scatter: colour by cluster assignment ───────────────────────────────
PLOT_SAMPLE = 15_000
plot_idx = rng.choice(len(cluster_labels), size=PLOT_SAMPLE, replace=False)

palette_clusters = cm.get_cmap("tab10", N_CLUSTERS)
palette_gestures = cm.get_cmap("tab10", 7)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: coloured by K-Means cluster
ax = axes[0]
for c in range(N_CLUSTERS):
    mask = cluster_labels[plot_idx] == c
    ax.scatter(
        X_2d[plot_idx][mask, 0], X_2d[plot_idx][mask, 1],
        s=2, alpha=0.4, color=palette_clusters(c), label=f"Cluster {c}"
    )
ax.set_title(f"PCA-2D — K-Means clusters (k={N_CLUSTERS})", fontsize=13)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.legend(markerscale=4, fontsize=8, loc="best")

# Right: coloured by true gesture label
ax = axes[1]
for g in range(7):
    mask = y[plot_idx] == g
    ax.scatter(
        X_2d[plot_idx][mask, 0], X_2d[plot_idx][mask, 1],
        s=2, alpha=0.4, color=palette_gestures(g), label=GESTURE_NAMES[g]
    )
ax.set_title("PCA-2D — True gesture labels", fontsize=13)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.legend(markerscale=4, fontsize=8, loc="best")

plt.tight_layout()
plt.savefig("scatter_pca2d.png", dpi=150)
plt.show()
print("Saved scatter_pca2d.png")

In [ ]:
# ── Confusion matrix: cluster (rows) vs true gesture (cols) ───────────────────
cm_matrix = confusion_matrix(cluster_labels, y, labels=list(range(N_CLUSTERS)))

# Normalise rows so each cluster sums to 1
cm_norm = cm_matrix.astype(float) / cm_matrix.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(
    cm_norm,
    annot=True, fmt=".2f", cmap="Blues",
    xticklabels=GESTURE_NAMES,
    yticklabels=[f"Cluster {c}" for c in range(N_CLUSTERS)],
    ax=ax,
)
ax.set_xlabel("True gesture label", fontsize=12)
ax.set_ylabel("K-Means cluster", fontsize=12)
ax.set_title(f"Cluster composition (row-normalised)  —  k={N_CLUSTERS}", fontsize=13)
plt.tight_layout()
plt.savefig("confusion_cluster_vs_gesture.png", dpi=150)
plt.show()
print("Saved confusion_cluster_vs_gesture.png")

In [ ]:
# ── Elbow plot: inertia for k = 2..12 (sanity check that k=8 is reasonable) ──
print("Computing elbow curve for k=2..12 (this may take ~1–2 min) ...")
ks = list(range(2, 13))
inertias = []

for k in ks:
    km = MiniBatchKMeans(
        n_clusters=k, n_init=5, random_state=RANDOM_STATE,
        batch_size=4096, max_iter=200
    )
    km.fit(X_pca)
    inertias.append(km.inertia_)
    print(f"  k={k:2d}  inertia={km.inertia_:.2e}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ks, inertias, marker="o", color="steelblue")
ax.axvline(N_CLUSTERS, color="red", linestyle="--", label=f"k={N_CLUSTERS} (chosen)")
ax.set_xlabel("Number of clusters k", fontsize=12)
ax.set_ylabel("Inertia (within-cluster SSE)", fontsize=12)
ax.set_title("Elbow plot", fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig("elbow_plot.png", dpi=150)
plt.show()
print("Saved elbow_plot.png")